# JIE Training Notebook

Train GPT-2 Medium and generate checkpoints for TracIn-based backdoor detection.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Install Dependencies

In [ ]:
!pip install torch>=2.0.0 transformers>=4.35.0 datasets>=2.14.0 accelerate>=0.23.0 -q

## 3. Clone Repository

In [ ]:
!git clone https://github.com/Jinendran10/Mini-Project.git
%cd Mini-Project

## 4. Load Dataset

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2-medium")
tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="train[:1%]")
print(f"Loaded {len(dataset)} samples")

In [ ]:
def tokenize_function(examples):
    outputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
        return_tensors=None
    )
    outputs["labels"] = outputs["input_ids"].copy()
    return outputs

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names
)
print(f"Tokenized {len(tokenized_dataset)} samples")

## 5. Setup Model and Training

In [ ]:
from transformers import GPT2LMHeadModel, TrainingArguments, Trainer
import torch

model = GPT2LMHeadModel.from_pretrained("gpt2-medium")
print(f"Model loaded: {model.num_parameters() / 1e6:.1f}M parameters")

checkpoint_dir = "/content/drive/MyDrive/jie_checkpoints"
!mkdir -p {checkpoint_dir}

training_args = TrainingArguments(
    output_dir=checkpoint_dir,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=100,
    save_steps=500,
    save_total_limit=3,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

## 6. Run Training

In [ ]:
trainer.train()

## 7. Save Final Model

In [ ]:
trainer.save_model(f"{checkpoint_dir}/final")
print(f"Training complete. Checkpoints saved to {checkpoint_dir}")

## 8. Test JIE Detection (Optional)

In [ ]:
import sys
sys.path.append("/content/Mini-Project/src")

from jie.detector import JIEDetector

samples = [
    {"sample_id": "test1", "text": "The capital of France is"},
    {"sample_id": "test2", "text": "Machine learning is"}
]

detector = JIEDetector(
    model_name="gpt2-medium",
    checkpoint_paths=[f"{checkpoint_dir}/checkpoint-500", f"{checkpoint_dir}/final"],
    device="cuda" if torch.cuda.is_available() else "cpu"
)

results = detector.detect(samples)
print("\nDetection Results:")
for r in results:
    print(f"{r['sample_id']}: score={r['tracin_score']:.6f}")